In [1]:
import pandas as pd
import numpy as np

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [2]:
# Cargar dataset diabetes
data = load_diabetes(as_frame=True)
df = data.frame

df.head()

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0


In [3]:
promedio = df["target"].mean()

df["diabetes_categoria"] = np.where(
    df["target"] >= promedio,
    1,
    0
)

df.head()

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target,diabetes_categoria
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0,0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0,0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0,0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0,1
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0,0


In [4]:
X = df.drop(columns=["target", "diabetes_categoria"])
y = df["diabetes_categoria"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [5]:
arbol_basico = DecisionTreeClassifier(random_state=42)

arbol_basico.fit(X_train, y_train)

pred_basico = arbol_basico.predict(X_test)

print("Precisión Árbol Básico:",
      accuracy_score(y_test, pred_basico))

print("\nMatriz de Confusión:")
print(confusion_matrix(y_test, pred_basico))

print("\nReporte de Clasificación:")
print(classification_report(y_test, pred_basico))

Precisión Árbol Básico: 0.6629213483146067

Matriz de Confusión:
[[38 12]
 [18 21]]

Reporte de Clasificación:
              precision    recall  f1-score   support

           0       0.68      0.76      0.72        50
           1       0.64      0.54      0.58        39

    accuracy                           0.66        89
   macro avg       0.66      0.65      0.65        89
weighted avg       0.66      0.66      0.66        89



In [6]:
# Parámetros a evaluar
param_grid = {
    "max_depth": [3, 5, 10],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 5, 10]
}

grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="accuracy"
)

grid_search.fit(X_train, y_train)

print("Mejores parámetros:")
print(grid_search.best_params_)

Mejores parámetros:
{'max_depth': 3, 'min_samples_leaf': 10, 'min_samples_split': 2}


In [7]:
# Evaluar árbol optimizado
mejor_arbol = grid_search.best_estimator_

pred_optimizado = mejor_arbol.predict(X_test)

print("Precisión Árbol Optimizado:",
      accuracy_score(y_test, pred_optimizado))

print("\nMatriz de Confusión:")
print(confusion_matrix(y_test, pred_optimizado))

print("\nReporte de Clasificación:")
print(classification_report(y_test, pred_optimizado))

Precisión Árbol Optimizado: 0.6741573033707865

Matriz de Confusión:
[[42  8]
 [21 18]]

Reporte de Clasificación:
              precision    recall  f1-score   support

           0       0.67      0.84      0.74        50
           1       0.69      0.46      0.55        39

    accuracy                           0.67        89
   macro avg       0.68      0.65      0.65        89
weighted avg       0.68      0.67      0.66        89



Conclusión: Se comparó un árbol de decisión básico con uno optimizado mediante GridSearchCV. La optimización permitió encontrar la mejor combinación de hiperparámetros, ayudando a reducir el sobreajuste y mejorar la capacidad de generalización del modelo. En este caso, el árbol optimizado obtuvo un mejor desempeño que el modelo inicial.